# Neutral-atom quantum computing for multi-hypothesis tracking

Run every cell top to bottom. The notebook first demonstrates one bounded
tracking sequence, then runs a resumable synthetic benchmark and plots it.

- A small, quantum-friendly synthetic sequence is used by default. Set
  `USE_QUANTUM_DEMO_DATA = False` to prefer an installed real sequence.
- The preset has eight low-noise frames and a strict eight-node solver cap;
  the default three-frame run includes a real non-clique simulation.
- Switch to the quantum solver in the configuration cell
  (requires `python -m pip install -e ".[quantum]"`).
- The overnight section independently varies motion, missed detections,
  clutter, sensor noise, object count, and random seed. It streams frames
  without filling the data directory and checkpoints every result to SQLite.
- `BENCHMARK_PROFILE = "overnight"` is ready for a long unattended run.
  Use `"smoke"` first if you want to verify the environment in minutes.

In [ ]:
# Imports: make the local src-layout package available before importing it.
from pathlib import Path
import sys

import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
source_root = project_root / "src"
if not (project_root / "pyproject.toml").is_file() or not source_root.is_dir():
    raise RuntimeError("Open user_notebook.ipynb from the repository root.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from neutral_atom_mht import (
    BENCHMARK_SCHEMA_VERSION,
    ClassicalSolver,
    DEFAULT_SYNTHETIC_DATA_ROOT,
    HPC,
    HPCConfig,
    OvernightBenchmarkConfig,
    QUANTUM_DEMO_DATA_CONFIG,
    QuantumSolver,
    SyntheticDataConfig,
    SyntheticDataGenerator,
    SyntheticDataset,
    build_synthetic_scenarios,
    run_overnight_benchmark,
)
from cell_data import DATASET_NAME, FRAME_COUNT, load_tiff, raw_frame_path

In [ ]:
# Data: use the bounded quantum demo by default. Set the flag to False to
# prefer an installed real sequence, with synthetic data as its fallback.
frame = 0
real_dataset_root = project_root / "data" / DATASET_NAME
USE_QUANTUM_DEMO_DATA = True
USE_SYNTHETIC_DATA = (
    USE_QUANTUM_DEMO_DATA
    or not raw_frame_path(real_dataset_root, frame).is_file()
)

# This versioned eight-frame preset keeps quantum components small while
# retaining one genuine non-clique Pulser simulation.
synthetic_data_config = QUANTUM_DEMO_DATA_CONFIG
synthetic_output_root = project_root / DEFAULT_SYNTHETIC_DATA_ROOT
synthetic_dataset = SyntheticDataset(
    root=synthetic_output_root / synthetic_data_config.dataset_name,
    config=synthetic_data_config,
)
if USE_SYNTHETIC_DATA and not synthetic_dataset.raw_frame_path(0).is_file():
    print(f"Real data not found; generating {synthetic_dataset.root} ...")
    synthetic_dataset = SyntheticDataGenerator(synthetic_data_config).generate(
        synthetic_output_root
    )
    print("Synthetic sequence generated.")

In [ ]:
# Configuration: resolve the selected data source and choose a solver.
if USE_SYNTHETIC_DATA:
    dataset_root = synthetic_dataset.root
    frame_path = synthetic_dataset.raw_frame_path(frame)
    sequence = synthetic_dataset.config.sequence
    dataset_label = synthetic_data_config.dataset_name
    available_frames = synthetic_data_config.frame_count
else:
    dataset_root = real_dataset_root
    frame_path = raw_frame_path(dataset_root, frame)
    sequence = "01"
    dataset_label = DATASET_NAME
    available_frames = FRAME_COUNT
print(f"Using {dataset_label} sequence {sequence}: {frame_path}")

config = HPCConfig()
controller = HPC(config, sequence=sequence)
# The quantum demo preset was validated with components of at most five nodes.
# solver = ClassicalSolver(maximum_component_nodes=60)
# Optional quantum simulation (requires `python -m pip install -e ".[quantum]"`).
# The explicit cap prevents accidental exponential simulation growth.
solver = QuantumSolver(maximum_component_nodes=8)

In [ ]:
# Run: detect the cells, create tracks, and show the detections on the source frame.
image = load_tiff(frame_path)
prepared = controller.prepare_frame(image, frame=frame)
solver_result = controller.solve(prepared, solver)
result = controller.advance(prepared, solver_result)
detections = prepared.observed_frame.detection.detections

figure, axis = plt.subplots(figsize=(10, 8))
axis.imshow(image, cmap="gray")
axis.scatter(
    [detection.x_px for detection in detections],
    [detection.y_px for detection in detections],
    s=28,
    facecolors="none",
    edgecolors="tab:red",
    linewidths=0.9,
    label="detected cell",
)
axis.set_title(
    f"{dataset_label} sequence {sequence}, frame {frame:03d}: "
    f"{len(detections)} detections"
)
axis.set_axis_off()
axis.legend(loc="upper right")
plt.show()

{
    "dataset": dataset_label,
    "frame_path": str(frame_path),
    "detections": len(detections),
    "initialized_tracks": len(result.tracks),
    "solver": solver.solver_name,
}

In [ ]:
# Optional multi-frame run: enable explicitly, especially before using QuantumSolver.
RUN_MANY_FRAMES = True
MANY_FRAME_COUNT = 3
sequence_summary = None
if RUN_MANY_FRAMES:
    frames_to_run = min(MANY_FRAME_COUNT, available_frames)
    if frames_to_run < 1:
        raise ValueError("MANY_FRAME_COUNT must be positive")

    if USE_SYNTHETIC_DATA:
        frame_paths = [
            synthetic_dataset.raw_frame_path(frame_index)
            for frame_index in range(frames_to_run)
        ]
    else:
        frame_paths = [
            raw_frame_path(dataset_root, frame_index)
            for frame_index in range(frames_to_run)
        ]
    images = (load_tiff(path) for path in frame_paths)

    sequence_controller = HPC(config, sequence=sequence)
    sequence_result = sequence_controller.run_sequence(
        images, solver, start_frame=0
    )
    processed_frames = [step.frame for step in sequence_result.steps]
    active_tracks = [len(step.tracks) for step in sequence_result.steps]
    assigned_observations = [
        len(step.assigned_observation_ids) for step in sequence_result.steps
    ]

    figure, axis = plt.subplots(figsize=(10, 4))
    axis.plot(processed_frames, active_tracks, label="active tracks")
    axis.plot(
        processed_frames,
        assigned_observations,
        label="assigned observations",
    )
    axis.set(
        xlabel="frame",
        ylabel="count",
        title=(
            f"{dataset_label}: {frames_to_run} frames with "
            f"{sequence_result.solver_name}"
        ),
    )
    axis.grid(alpha=0.25)
    axis.legend()
    plt.show()

    sequence_summary = {
        "frames_processed": len(sequence_result.steps),
        "final_tracks": len(sequence_result.final_tracks),
        "solver": sequence_result.solver_name,
        "solver_runtime_seconds": sum(
            step.solver_result.runtime_seconds for step in sequence_result.steps
        ),
    }
sequence_summary

## Resumable synthetic benchmark

This campaign separates motion, missed detections, clutter, and sensor noise
instead of hiding them behind one coupled setting. It also varies density and
random seed. Frames are streamed in memory, exact results are screened first,
and neutral-atom simulations are stratified by difficulty axis, severity, and
supported non-clique component size. Every completed frame is committed to
SQLite, so interrupting
the kernel is safe: rerun these cells to resume and refresh the tidy CSV export.
The configured limit budgets newly checkpointed work, not wall-clock time:
replay and export are excluded, and an individual solver call is not interrupted.

In [ ]:
# Choose "smoke" for a short environment check or "overnight" for the full run.
BENCHMARK_PROFILE = "overnight"
BENCHMARK_AXES = ("motion", "dropout", "clutter", "sensor_noise", "combined")
if BENCHMARK_PROFILE == "smoke":
    BENCHMARK_SEVERITIES = (0.5, 1.0)
    BENCHMARK_OBJECT_COUNTS = (4,)
    BENCHMARK_SEEDS = (0,)
    BENCHMARK_FRAME_COUNT = 4
    QUANTUM_QUOTA_PER_STRATUM = 1
    BENCHMARK_FORWARD_WORK_HOURS = 0.25
elif BENCHMARK_PROFILE == "overnight":
    BENCHMARK_SEVERITIES = (0.2, 0.4, 0.6, 0.8, 1.0)
    BENCHMARK_OBJECT_COUNTS = (4, 12, 30, 55)
    BENCHMARK_SEEDS = tuple(range(5))
    BENCHMARK_FRAME_COUNT = 40
    QUANTUM_QUOTA_PER_STRATUM = 5
    BENCHMARK_FORWARD_WORK_HOURS = 10.0
else:
    raise ValueError('BENCHMARK_PROFILE must be "smoke" or "overnight"')

benchmark_scenarios = build_synthetic_scenarios(
    axes=("baseline", *BENCHMARK_AXES),
    severity_levels=BENCHMARK_SEVERITIES,
    object_counts=BENCHMARK_OBJECT_COUNTS,
    seeds=BENCHMARK_SEEDS,
    frame_count=BENCHMARK_FRAME_COUNT,
    image_shape=QUANTUM_DEMO_DATA_CONFIG.image_shape,
)
overnight_config = OvernightBenchmarkConfig(
    scenarios=benchmark_scenarios,
    output_directory=(
        project_root
        / "outputs"
        / "overnight"
        / f"schema-{BENCHMARK_SCHEMA_VERSION}"
        / BENCHMARK_PROFILE
    ),
    exact_maximum_component_nodes=128,  # Avoid truncating the 55-object cells.
    run_quantum=True,
    quantum_max_nonclique_component_nodes=8,
    quantum_quota_per_stratum=QUANTUM_QUOTA_PER_STRATUM,
    store_detailed_records=False,  # Keep the overnight database compact.
    forward_work_budget_seconds=BENCHMARK_FORWARD_WORK_HOURS * 60.0 * 60.0,
    progress_every_frames=100,
    resume=True,
)
{
    "profile": BENCHMARK_PROFILE,
    "scenarios": len(benchmark_scenarios),
    "planned_exact_frames": len(benchmark_scenarios) * BENCHMARK_FRAME_COUNT,
    "quantum_candidates_per_axis_severity_size_stratum": (
        QUANTUM_QUOTA_PER_STRATUM
    ),
    "store_detailed_records": overnight_config.store_detailed_records,
    "forward_work_budget_hours": BENCHMARK_FORWARD_WORK_HOURS,
    "output_directory": str(overnight_config.output_directory),
}

In [ ]:
# Long-running cell: progress is durable after every frame and reruns resume.
overnight_result = run_overnight_benchmark(
    overnight_config, progress=lambda update: print(update, flush=True)
)
benchmark_records = list(overnight_result.records)
benchmark_summary = dict(overnight_result.summary)
benchmark_summary.update(
    {
        "database": str(overnight_result.database_path),
        "csv": str(overnight_result.csv_path),
        "stopped_reason": overnight_result.stopped_reason,
    }
)
benchmark_summary

## Publication figures

The following cells save eight reproducible figures under `outputs/figures/`.
Figures 1–4 explain the workflow and retain the validated frame-2 quantum
example. Figures 5–8 summarize every checkpoint currently available from the
broad campaign, even when its forward-work budget stopped a partial run. Cell
annotations expose checkpoint and sample counts; partial or differently sampled
cells are descriptive rather than balanced comparisons. Quantum
and exact scores always use the same immutable frame graph; the exact result
alone advances the reference trajectory. Missing, oversized, or failed quantum
runs remain visible as coverage/status results instead of aborting the notebook.

In [ ]:
# Shared publication-figure imports, output path, and rendering helpers.
from itertools import combinations

import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch

from graph import logical_layout

figure_output_dir = project_root / "outputs" / "figures"
figure_output_dir.mkdir(parents=True, exist_ok=True)

def finish_figure(figure, filename):
    path = figure_output_dir / filename
    figure.tight_layout()
    figure.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    return path

def draw_detection_overlay(axis, image, detection, title):
    axis.imshow(image, cmap="gray")
    if np.any(detection.labels):
        axis.contour(detection.labels > 0, levels=[0.5], colors="#00D4FF", linewidths=0.7)
    axis.scatter(
        [item.x_px for item in detection.detections],
        [item.y_px for item in detection.detections],
        s=42, facecolors="none", edgecolors="#FF4D4D", linewidths=1.2,
    )
    axis.set_title(f"{title} ({len(detection.detections)} detections)")
    axis.set_axis_off()

def finite_values(rows, key):
    values = []
    for row in rows:
        value = row.get(key)
        if value is not None and np.isfinite(float(value)):
            values.append(float(value))
    return np.asarray(values, dtype=float)

def matching_rows(rows, row_value, column_value):
    return [
        row for row in rows
        if row["axis"] == row_value
        and np.isclose(row["severity"], column_value)
    ]

def aggregate_matrix(rows, row_values, column_values, value_key, reducer=np.median):
    matrix = np.full((len(row_values), len(column_values)), np.nan)
    for row_index, row_value in enumerate(row_values):
        for column_index, column_value in enumerate(column_values):
            selected = matching_rows(rows, row_value, column_value)
            values = finite_values(selected, value_key)
            if values.size:
                matrix[row_index, column_index] = reducer(values)
    return matrix

def count_matrix(rows, row_values, column_values):
    matrix = np.zeros((len(row_values), len(column_values)), dtype=int)
    for row_index, row_value in enumerate(row_values):
        for column_index, column_value in enumerate(column_values):
            matrix[row_index, column_index] = len(
                matching_rows(rows, row_value, column_value)
            )
    return matrix

def draw_heatmap(axis, matrix, row_labels, column_labels, *, title, colorbar_label, cmap="viridis", vmin=None, vmax=None, value_format=".2f", annotations=None):
    image = axis.imshow(matrix, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set_xticks(range(len(column_labels)), column_labels)
    axis.set_yticks(range(len(row_labels)), row_labels)
    axis.set_xlabel("difficulty severity")
    axis.set_title(title)
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[row_index, column_index]
            label = format(value, value_format) if np.isfinite(value) else "—"
            if annotations is not None and annotations[row_index, column_index]:
                label += f"\n{annotations[row_index, column_index]}"
            rgba = image.cmap(image.norm(value)) if np.isfinite(value) else (1, 1, 1, 1)
            luminance = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
            text_color = "white" if luminance < 0.48 else "black"
            axis.text(column_index, row_index, label, ha="center", va="center", fontsize=8, color=text_color)
    axis.figure.colorbar(image, ax=axis, label=colorbar_label)
    return image

In [ ]:
# Fig. 1 — End-to-end image, solver, and tracking workflow.
figure, axis = plt.subplots(figsize=(16, 4.2))
axis.set_xlim(0, 1)
axis.set_ylim(0, 1)
axis.axis("off")

workflow_boxes = [
    (0.01, 0.43, 0.11, 0.25, "Image frame", "#D9EAF7"),
    (0.15, 0.43, 0.11, 0.25, "Detect\nobservations", "#D9EAF7"),
    (0.29, 0.43, 0.13, 0.25, "Predict, gate,\nand weight", "#FCE8C3"),
    (0.45, 0.43, 0.12, 0.25, "Weighted\nconflict graph", "#FCE8C3"),
    (0.60, 0.35, 0.14, 0.41, "Solver\n\nClassical exact\nor\nneutral atom:\nembed → pulse → sample", "#E4D7F5"),
    (0.77, 0.43, 0.10, 0.25, "Selected\nassociations", "#D7F0E3"),
    (0.90, 0.43, 0.09, 0.25, "Bayesian +\nKalman update", "#D7F0E3"),
]
for x, y, width, height, label, color in workflow_boxes:
    box = FancyBboxPatch(
        (x, y), width, height, boxstyle="round,pad=0.012",
        facecolor=color, edgecolor="#263238", linewidth=1.4,
    )
    axis.add_patch(box)
    axis.text(x + width / 2, y + height / 2, label, ha="center", va="center", fontsize=10)
for left, right in zip(workflow_boxes, workflow_boxes[1:]):
    start = (left[0] + left[2], left[1] + left[3] / 2)
    stop = (right[0], right[1] + right[3] / 2)
    axis.add_patch(FancyArrowPatch(start, stop, arrowstyle="-|>", mutation_scale=14, color="#455A64"))
axis.add_patch(FancyArrowPatch(
    (0.945, 0.42), (0.355, 0.42), connectionstyle="arc3,rad=-0.35",
    arrowstyle="-|>", mutation_scale=14, color="#00796B", linewidth=1.6,
))
axis.text(0.66, 0.06, "retained tracks become the next frame's prior state", ha="center", color="#00796B")
axis.set_title("Fig. 1 — Neutral-atom multi-hypothesis tracking workflow", fontsize=14, pad=12)
fig1_path = finish_figure(figure, "fig1_workflow.png")
fig1_path

In [ ]:
# Fig. 2 — Real and simulated detection overlays.
synthetic_figure_dataset = synthetic_dataset
if not synthetic_figure_dataset.raw_frame_path(0).is_file():
    synthetic_figure_dataset = SyntheticDataGenerator(synthetic_data_config).generate(
        synthetic_output_root
    )
synthetic_image = synthetic_figure_dataset.load_frame(0)
synthetic_detection = HPC(config, sequence=synthetic_data_config.sequence).observe(
    synthetic_image, frame=0
).detection

figure, axes = plt.subplots(1, 2, figsize=(14, 5.5))
real_path = raw_frame_path(real_dataset_root, 0)
if real_path.is_file():
    real_image = load_tiff(real_path)
    real_detection = HPC(config, sequence="01").observe(real_image, frame=0).detection
    draw_detection_overlay(axes[0], real_image, real_detection, "Real data, frame 000")
else:
    axes[0].set_facecolor("#F3F4F6")
    axes[0].text(0.5, 0.56, "Real sequence not installed", ha="center", va="center", fontsize=13)
    axes[0].text(0.5, 0.44, str(real_path), ha="center", va="center", fontsize=8, color="#666666", wrap=True)
    axes[0].set_title("Real data, frame 000")
    axes[0].set_xticks([])
    axes[0].set_yticks([])
draw_detection_overlay(
    axes[1], synthetic_image, synthetic_detection,
    f"Simulated {synthetic_data_config.dataset_name}, frame 000",
)
figure.suptitle("Fig. 2 — Image detection overlays", fontsize=14)
fig2_path = finish_figure(figure, "fig2_detection_overlays.png")
{"path": fig2_path, "real_data_available": real_path.is_file()}

In [ ]:
# Reproduce the validated frame-2 example used only by Figs. 3 and 4.
# The large campaign above supplies every aggregate performance figure.
example_reference = HPC(HPCConfig(), sequence=QUANTUM_DEMO_DATA_CONFIG.sequence)
example_exact_solver = ClassicalSolver(maximum_component_nodes=8)
example_quantum_solver = QuantumSolver(maximum_component_nodes=8)
example_prepared = None
example_execution = None
example_exact_result = None
for example_frame, (example_image, _) in enumerate(
    SyntheticDataGenerator(QUANTUM_DEMO_DATA_CONFIG).iter_frames()
):
    prepared_frame = example_reference.prepare_frame(
        example_image, frame=example_frame
    )
    exact_result = example_exact_solver.solve(prepared_frame.solver_input())
    if not exact_result.successful:
        raise RuntimeError(f"Validated exact example failed at frame {example_frame}")
    if example_frame == 2:
        example_prepared = prepared_frame
        example_exact_result = exact_result
        example_execution = example_quantum_solver.execute(
            prepared_frame.solver_input()
        )
    example_reference.advance(prepared_frame, exact_result)
    if example_frame == 2:
        break

if example_prepared is None or example_execution is None or not example_execution.successful:
    raise RuntimeError("The validated frame-2 quantum example was not produced")
simulated_example_runs = tuple(
    run for run in example_execution.runs if run.program is not None
)
if not simulated_example_runs:
    raise RuntimeError(
        "Frame 2 contains no simulated component; regenerate the validated preset"
    )
example_run = max(
    simulated_example_runs,
    key=lambda run: (len(run.node_ids), -run.component_id),
)
{
    "frame": example_prepared.frame,
    "graph_nodes": len(example_prepared.graph.nodes),
    "quantum_status": example_execution.status,
    "selection_agrees": (
        example_execution.selected_ids == example_exact_result.selected_ids
    ),
}

In [ ]:
# Fig. 3 — Logical embedding of the frame-2 weighted conflict graph.
example_graph = example_prepared.graph
graph_positions = logical_layout(example_graph)
quantum_selected = set(example_execution.selected_ids)
figure, axis = plt.subplots(figsize=(9, 6))
for left, right in example_graph.edges:
    axis.plot(
        [graph_positions[left][0], graph_positions[right][0]],
        [graph_positions[left][1], graph_positions[right][1]],
        color="#8795A1", linewidth=1.8, zorder=1,
    )
for node in example_graph.nodes:
    x_value, y_value = graph_positions[node.node_id]
    selected = node.node_id in quantum_selected
    axis.scatter(
        [x_value], [y_value], s=1100,
        color="#2A9D8F" if selected else "#E9C46A",
        edgecolor="#173F5F", linewidth=2.5 if selected else 1.2, zorder=2,
    )
    axis.text(
        x_value, y_value,
        f"h{node.node_id}\nT{node.track_id}→O{node.observation_id}\nw={node.weight:.2f}",
        ha="center", va="center", fontsize=8, zorder=3,
    )
axis.set_title("Fig. 3 — Frame 002 weighted association-conflict graph")
axis.text(0.5, -0.04, "edge = shared track or observation; teal = quantum-selected hypothesis", transform=axis.transAxes, ha="center")
axis.set_aspect("equal", adjustable="datalim")
axis.axis("off")
fig3_path = finish_figure(figure, "fig3_conflict_graph.png")
fig3_path

In [ ]:
# Fig. 4 — Stylized physical neutral-atom representation of that component.
program = example_run.program
if program is None:
    raise RuntimeError("The selected example has no neutral-atom program")
component = program.component
coordinates = dict(zip(example_run.node_ids, example_run.coordinates, strict=True))
qubit_by_node = dict(zip(component.node_ids, component.qubit_ids, strict=True))
blockade_distance = program.sequence.device.rydberg_blockade_radius(program.omega)
selected_atoms = set(example_run.selected_ids)
intended_edges = {tuple(sorted(edge)) for edge in component.edges}
physical_edges = set()
for left, right in combinations(component.node_ids, 2):
    separation = np.linalg.norm(
        np.asarray(coordinates[left]) - np.asarray(coordinates[right])
    )
    if separation <= blockade_distance:
        physical_edges.add(tuple(sorted((left, right))))

figure, axis = plt.subplots(figsize=(8, 7))
for node_id, (x_value, y_value) in coordinates.items():
    axis.add_patch(Circle(
        (x_value, y_value), blockade_distance / 2.0,
        facecolor="#4CC9F0", edgecolor="#168AAD", alpha=0.08, linewidth=1.0,
    ))
for left, right in sorted(intended_edges | physical_edges):
    if (left, right) in intended_edges and (left, right) in physical_edges:
        color, linestyle, linewidth = "#2A9D8F", "-", 2.3
    elif (left, right) in intended_edges:
        color, linestyle, linewidth = "#D62828", "--", 2.0
    else:
        color, linestyle, linewidth = "#7B2CBF", ":", 2.0
    axis.plot(
        [coordinates[left][0], coordinates[right][0]],
        [coordinates[left][1], coordinates[right][1]],
        color=color, linestyle=linestyle, linewidth=linewidth, zorder=1,
    )
for node_id in component.node_ids:
    x_value, y_value = coordinates[node_id]
    selected = node_id in selected_atoms
    axis.scatter(
        [x_value], [y_value], s=430,
        color="#E63946" if selected else "#F4A261",
        edgecolor="#1D3557", linewidth=1.6, zorder=3,
    )
    axis.text(
        x_value, y_value, f"{qubit_by_node[node_id]}\nh{node_id}",
        ha="center", va="center", fontsize=8, color="white", zorder=4,
    )
legend_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#E63946", markeredgecolor="#1D3557", markersize=11, label="selected Rydberg atom"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#F4A261", markeredgecolor="#1D3557", markersize=11, label="unselected atom"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#4CC9F0", alpha=0.25, markersize=14, label="$R_b/2$ interaction halo"),
]
if intended_edges & physical_edges:
    legend_handles.append(Line2D([0], [0], color="#2A9D8F", linewidth=2.3, label="intended edge realized by blockade"))
if intended_edges - physical_edges:
    legend_handles.append(Line2D([0], [0], color="#D62828", linestyle="--", linewidth=2.0, label="intended edge outside blockade"))
if physical_edges - intended_edges:
    legend_handles.append(Line2D([0], [0], color="#7B2CBF", linestyle=":", linewidth=2.0, label="unintended blockade pair"))
axis.legend(handles=legend_handles, loc="upper right")
realized_count = len(intended_edges & physical_edges)
axis.set(
    xlabel="x coordinate (μm)", ylabel="y coordinate (μm)",
    title=("Fig. 4 — Heuristic neutral-atom embedding of the simulated component\n"
           f"mapping cost={example_run.mapping_cost:.3f}; "
           f"realized intended edges={realized_count}/{len(intended_edges)}; "
           f"selected hypotheses={example_run.selected_ids}"),
)
axis.set_aspect("equal", adjustable="datalim")
axis.grid(alpha=0.15)
fig4_path = finish_figure(figure, "fig4_neutral_atom_embedding.png")
fig4_path

In [ ]:
# Fig. 5 — Descriptive quantum objective ratios with explicit sample coverage.
plot_scenarios = ("baseline", *BENCHMARK_AXES)
plot_severities = (0.0, *BENCHMARK_SEVERITIES)
planned_frames_per_axis_cell = (
    len(BENCHMARK_OBJECT_COUNTS) * len(BENCHMARK_SEEDS) * BENCHMARK_FRAME_COUNT
)
successful_quantum_rows = [
    row for row in benchmark_records
    if row.get("quantum_status") == "completed"
    and row.get("relative_objective") is not None
]
objective_ratio_matrix = aggregate_matrix(
    successful_quantum_rows, plot_scenarios, plot_severities, "relative_objective"
)
quantum_n_matrix = count_matrix(
    successful_quantum_rows, plot_scenarios, plot_severities
)
prepared_benchmark_rows = [
    row for row in benchmark_records if row.get("input_fingerprint") is not None
]
checkpoint_n_matrix = count_matrix(
    prepared_benchmark_rows, plot_scenarios, plot_severities
)
objective_annotations = np.full(objective_ratio_matrix.shape, "", dtype=object)
for row_index, scenario_name in enumerate(plot_scenarios):
    for severity_index, severity in enumerate(plot_severities):
        valid_cell = (
            scenario_name == "baseline" and np.isclose(severity, 0.0)
        ) or (scenario_name != "baseline" and not np.isclose(severity, 0.0))
        if valid_cell:
            objective_annotations[row_index, severity_index] = (
                f"q={quantum_n_matrix[row_index, severity_index]}; "
                f"seen={checkpoint_n_matrix[row_index, severity_index]}/"
                f"{planned_frames_per_axis_cell}"
            )
finite_ratios = objective_ratio_matrix[np.isfinite(objective_ratio_matrix)]
ratio_floor = min(0.95, float(finite_ratios.min())) if finite_ratios.size else 0.0
figure, axis = plt.subplots(figsize=(11, 7.0))
draw_heatmap(
    axis, objective_ratio_matrix, plot_scenarios, plot_severities,
    title="Fig. 5 — Descriptive median neutral-atom / exact objective",
    colorbar_label="median objective ratio", cmap="viridis",
    vmin=ratio_floor, vmax=1.0, value_format=".3f",
    annotations=objective_annotations,
)
axis.set_ylabel("synthetic difficulty axis")
axis.text(
    0.99, -0.16,
    "q = successful axis/severity/size-stratified samples; "
    "seen = prepared/planned exact frames.\n"
    "Cell medians are descriptive: q is not balanced by density, seed, or frame.",
    transform=axis.transAxes, ha="right", fontsize=8, color="#444444",
)
fig5_path = finish_figure(figure, "fig5_objective_ratio_heatmap.png")
fig5_path

In [ ]:
# Fig. 6 — Quantum-size eligibility across all prepared frames.
eligibility_matrix = np.full(
    (len(BENCHMARK_OBJECT_COUNTS), len(plot_severities)), np.nan
)
eligibility_annotations = np.full(eligibility_matrix.shape, "", dtype=object)
planned_frames_per_density_cell = len(BENCHMARK_SEEDS) * BENCHMARK_FRAME_COUNT
for object_index, object_count in enumerate(BENCHMARK_OBJECT_COUNTS):
    for severity_index, severity in enumerate(plot_severities):
        scenario_name = "baseline" if np.isclose(severity, 0.0) else "combined"
        rows = [
            row for row in benchmark_records
            if row["axis"] == scenario_name
            and row["object_count"] == object_count
            and np.isclose(row["severity"], severity)
            and row.get("input_fingerprint") is not None
        ]
        if rows:
            eligible = [
                0 < row["maximum_nonclique_component_nodes"]
                <= overnight_config.quantum_max_nonclique_component_nodes
                for row in rows
            ]
            eligible_count = int(sum(eligible))
            eligibility_matrix[object_index, severity_index] = (
                100.0 * eligible_count / len(rows)
            )
            eligibility_annotations[object_index, severity_index] = (
                f"{eligible_count}/{len(rows)} eligible\n"
                f"seen={len(rows)}/{planned_frames_per_density_cell}"
            )
figure, axis = plt.subplots(figsize=(10.5, 6.2))
draw_heatmap(
    axis, eligibility_matrix, BENCHMARK_OBJECT_COUNTS, plot_severities,
    title="Fig. 6 — Prepared frames eligible for neutral-atom simulation",
    colorbar_label="eligible prepared frames (%)", cmap="magma",
    vmin=0.0, vmax=100.0, value_format=".0f",
    annotations=eligibility_annotations,
)
axis.set_ylabel("synthetic object count")
axis.text(
    0.99, -0.15,
    "Denominator = every checkpointed prepared frame with component metrics, "
    "including non-optimal exact statuses; size 0 needs no non-clique simulation.",
    transform=axis.transAxes, ha="right", fontsize=8, color="#444444",
)
fig6_path = finish_figure(figure, "fig6_quantum_eligibility_heatmap.png")
fig6_path

In [ ]:
# Fig. 7 — Detection quality and identity-aware tracking outcomes.
figure, (quality_axis, event_axis) = plt.subplots(1, 2, figsize=(17, 6.5))
colors = plt.get_cmap("tab10")(np.linspace(0.0, 0.8, len(BENCHMARK_AXES)))
tracking_rows = [
    row for row in benchmark_records
    if row.get("tracking_metrics_status") == "completed"
]
latest_tracking_by_scenario = {}
for row in tracking_rows:
    previous = latest_tracking_by_scenario.get(row["scenario"])
    if previous is None or row["frame"] > previous["frame"]:
        latest_tracking_by_scenario[row["scenario"]] = row
final_tracking_rows = list(latest_tracking_by_scenario.values())

quality_metrics = (
    ("detection_recall", "detection recall", "-", "o"),
    ("matched_track_gt_identity_correctness", "track identity correctness", "--", "s"),
)
event_metrics = (
    ("cumulative_id_switch_count", "cumulative ID switches", "-", "o"),
    ("cumulative_fragmentation_count", "cumulative fragmentations", "--", "s"),
)
for scenario_name, color in zip(BENCHMARK_AXES, colors, strict=True):
    for metric_key, _, linestyle, marker in quality_metrics:
        medians = []
        source_rows = (
            prepared_benchmark_rows
            if metric_key == "detection_recall"
            else tracking_rows
        )
        baseline_source = [row for row in source_rows if row["axis"] == "baseline"]
        for severity in plot_severities:
            rows = baseline_source if np.isclose(severity, 0.0) else [
                row for row in source_rows
                if row["axis"] == scenario_name
                and np.isclose(row["severity"], severity)
            ]
            values = finite_values(rows, metric_key)
            medians.append(np.median(values) if values.size else np.nan)
        quality_axis.plot(
            plot_severities, medians, color=color, linestyle=linestyle,
            marker=marker, linewidth=1.8, markersize=5,
        )
    for metric_key, _, linestyle, marker in event_metrics:
        medians = []
        baseline_final = [
            row for row in final_tracking_rows if row["axis"] == "baseline"
        ]
        for severity in plot_severities:
            rows = baseline_final if np.isclose(severity, 0.0) else [
                row for row in final_tracking_rows
                if row["axis"] == scenario_name
                and np.isclose(row["severity"], severity)
            ]
            values = finite_values(rows, metric_key)
            medians.append(np.median(values) if values.size else np.nan)
        event_axis.plot(
            plot_severities, medians, color=color, linestyle=linestyle,
            marker=marker, linewidth=1.8, markersize=5,
        )
quality_axis.set(
    xlabel="difficulty severity", ylabel="median fraction",
    ylim=(-0.02, 1.02), title="Detection recall and track identity correctness",
)
event_axis.set(
    xlabel="difficulty severity", ylabel="median cumulative events",
    title="ID switches and fragmentations at latest checkpoint",
)
for plot_axis in (quality_axis, event_axis):
    plot_axis.grid(alpha=0.25)
axis_handles = [
    Line2D([0], [0], color=color, linewidth=2, label=scenario_name)
    for scenario_name, color in zip(BENCHMARK_AXES, colors, strict=True)
]
axis_legend = quality_axis.legend(
    handles=axis_handles, title="difficulty axis", ncol=2, loc="lower left"
)
quality_axis.add_artist(axis_legend)
quality_axis.legend(
    handles=[
        Line2D([0], [0], color="#333333", linestyle=linestyle, marker=marker, label=label)
        for _, label, linestyle, marker in quality_metrics
    ],
    loc="upper right",
)
event_axis.legend(
    handles=[
        Line2D([0], [0], color="#333333", linestyle=linestyle, marker=marker, label=label)
        for _, label, linestyle, marker in event_metrics
    ],
    loc="upper left",
)
figure.suptitle("Fig. 7 — Detection and identity-aware tracking quality", fontsize=14)
figure.text(
    0.99, 0.01,
    "Cumulative events use each scenario's latest successful checkpoint; "
    "partial scenarios have unequal exposure (see Fig. 5 seen counts).",
    ha="right", fontsize=8, color="#444444",
)
fig7_path = finish_figure(figure, "fig7_detection_and_tracking_quality.png")
fig7_path

In [ ]:
# Fig. 8 — Successful solver runtime scaling with connected-component size.
figure, axis = plt.subplots(figsize=(10.5, 6))
runtime_series = (
    ("classical exact", "maximum_component_nodes", "exact_runtime_seconds", ("exact_status", "optimal"), "#264653", "o"),
    ("neutral-atom quantum", "maximum_nonclique_component_nodes", "quantum_runtime_seconds", ("quantum_status", "completed"), "#E76F51", "s"),
)
runtime_success_counts = {}
for label, size_key, runtime_key, status_filter, color, marker in runtime_series:
    status_key, successful_status = status_filter
    successful_rows = [
        row for row in benchmark_records
        if row.get(status_key) == successful_status
        and row.get(size_key) is not None
        and row.get(runtime_key) not in (None, 0)
    ]
    runtime_success_counts[label] = len(successful_rows)
    sizes = sorted({
        int(row[size_key]) for row in successful_rows
    })
    medians, lower, upper, plotted_sizes = [], [], [], []
    for size in sizes:
        values = finite_values(
            [row for row in successful_rows if row.get(size_key) == size], runtime_key
        )
        values = values[values > 0.0]
        if values.size:
            plotted_sizes.append(size)
            medians.append(np.median(values))
            lower.append(np.quantile(values, 0.25))
            upper.append(np.quantile(values, 0.75))
    if plotted_sizes:
        axis.plot(plotted_sizes, medians, marker=marker, linewidth=2, color=color, label=label)
        axis.fill_between(plotted_sizes, lower, upper, color=color, alpha=0.15)
axis.set(
    xlabel="largest relevant component (nodes)", ylabel="solver runtime (seconds, log scale)",
    title="Fig. 8 — Successful solver runtime by component size (median and IQR)",
)
axis.set_yscale("log")
axis.grid(alpha=0.25, which="both")
handles, labels = axis.get_legend_handles_labels()
if handles:
    axis.legend()
axis.text(
    0.99, -0.13,
    "; ".join(f"{label}: n={count:,}" for label, count in runtime_success_counts.items()),
    transform=axis.transAxes, ha="right", fontsize=8, color="#444444",
)
fig8_path = finish_figure(figure, "fig8_runtime_by_component_size.png")
fig8_path